In [1]:
import polars as pl 
from pathlib import Path
from unidecode import unidecode

BASE_DIR = Path.cwd().parent / 'data_input' 


### limpeza e agregação dos dados OPTANTES

In [3]:
# dados_optantes = pl.read_excel(f'{BASE_DIR}/PROGRAMA_OPTANTES.xlsx', sheet_name='OPTANTES_04_20261')
# remuneracao_optantes = pl.read_excel(f'{BASE_DIR}/PROGRAMA_OPTANTES.xlsx', sheet_name='REMUNERAÇÃO_04_2026')
# base_ativos = pl.read_excel(f'{BASE_DIR}/PROGRAMA_OPTANTES.xlsx', sheet_name='BASE_ATIVOS_04_2026')
# resul_final = pl.read_excel(f'{BASE_DIR}/PROGRAMA_OPTANTES.xlsx', sheet_name='OPTANTES_04_2026_COBRANÇA')

### Testes Arquivos CRU

In [162]:
optantes_cru = pl.read_excel(f'{BASE_DIR}/OPTANTES_CRU.xlsx', sheet_name='Export') 
base_ativos_cru = pl.read_excel(f'{BASE_DIR}/BASE_ATIVOS_CANCELADOS_CRU.xlsx')
remuneracao_cru = pl.read_excel(f'{BASE_DIR}/REMUNERACAO_CRU.xlsx')
dados_copart = pl.read_excel(f'{BASE_DIR}/ANALITICO_COPARTICIPACAO_SIGRH.xlsx')

Could not determine dtype for column 23, falling back to string
Could not determine dtype for column 28, falling back to string


In [163]:
def formata_coluna(cols:list) -> list:
    cols_norm = []
    for col in cols:
        cols_norm.append(unidecode(col.lower().replace(' ', '_')))

    return cols_norm

optantes_cru.columns = formata_coluna(optantes_cru.columns)
base_ativos_cru.columns = formata_coluna(base_ativos_cru.columns)
remuneracao_cru.columns = formata_coluna(remuneracao_cru.columns)
dados_copart.columns = formata_coluna(dados_copart.columns)

In [164]:
remuneracao_cru = remuneracao_cru.with_columns([
    ((4 * pl.col('remuneracao_inas')) / 100).round(2).alias('valor_mensalidade')
]).with_columns([
    pl.when(pl.col('valor_mensalidade') < 578).then(578)
    .when(pl.col('valor_mensalidade') > 1538).then(1538)
    .otherwise('valor_mensalidade').alias('valor_mensalidade')
])

---

In [ ]:
optantes = (optantes_cru.join(
    base_ativos_cru[['cpf_titular', 'cpf_segurado', 'nome_segurado', 'codigo_da_empresa', 'matricula_titular_8_digitos', 'idade']], 
    on=['cpf_titular', 'cpf_segurado'], 
    how='inner')
    .drop(['nome_segurado_right', 'cpf_sem_ponto'])
    .sort(by='cpf_segurado')
    .rename({'matricula_titular_8_digitos':'matricula'})
    .with_columns([pl.col('codigo_da_empresa').cast(pl.Int64)])
    )


In [171]:
optantes = optantes.with_columns([
    pl.when(pl.col('tipo_beneficiario') == 'TITULAR').then(pl.lit('41134'))
    .when(pl.col('idade') <= 18).then(pl.lit('41135'))
    .when(pl.col('idade') <= 23).then(pl.lit('41171'))
    .when(pl.col('idade') <= 28).then(pl.lit('41175'))
    .when(pl.col('idade') <= 33).then(pl.lit('41191'))
    .when(pl.col('idade') <= 38).then(pl.lit('41192'))
    .when(pl.col('idade') <= 43).then(pl.lit('41193'))
    .when(pl.col('idade') <= 48).then(pl.lit('41121'))
    .when(pl.col('idade') <= 53).then(pl.lit('41122'))
    .when(pl.col('idade') <= 58).then(pl.lit('41123'))
    .otherwise(pl.lit('41224')).alias('rubrica')
]).with_columns([
    pl.when(pl.col('rubrica') == '41134').then(0)
    .when(pl.col('rubrica') == '41135').then(225)
    .when(pl.col('rubrica') == '41171').then(250)
    .when(pl.col('rubrica') == '41175').then(340)
    .when(pl.col('rubrica') == '41191').then(370)
    .when(pl.col('rubrica') == '41192').then(420)
    .when(pl.col('rubrica') == '41193').then(479)
    .when(pl.col('rubrica') == '41121').then(530)
    .when(pl.col('rubrica') == '41122').then(657)
    .when(pl.col('rubrica') == '41123').then(764)
    .otherwise(853).alias('mensalidade')
]).with_columns([
    pl.when(pl.col('situacao_funcional') == 'Titular optante anterior')
    .then('mensalidade')
    .otherwise(pl.col('mensalidade') * 2).alias('mensalidade')
])

In [175]:
optantes2 = optantes.group_by(['nome_segurado', 'cpf_titular', 'tipo_beneficiario', 'codigo_da_empresa', 'matricula', 'situacao_funcional']).agg(
    pl.col('mensalidade').sum()
)

In [180]:
pivot = optantes2.pivot(
    values='mensalidade',
    index=['cpf_titular','nome_segurado', 'codigo_da_empresa', 'matricula', 'situacao_funcional'],
    on='tipo_beneficiario'
).fill_null(0)

In [182]:
group2 = pivot.group_by('cpf_titular', 'codigo_da_empresa', 'matricula', 'situacao_funcional').agg(
    pl.col('TITULAR').sum().alias('mensalidade_titular'),
    pl.col('DEPENDENTE').sum().alias('mensalidade_dependente')
).filter(pl.col('codigo_da_empresa') != 911)

In [195]:
mensalidade = (group2.join(remuneracao_cru[['cpf', 'valor_mensalidade']], 
            left_on='cpf_titular', right_on='cpf', how='left')
      ).unique(subset='cpf_titular').fill_null(578)

final_optantes =  mensalidade.with_columns([
    pl.when(pl.col('situacao_funcional') == 'Titular optante anterior').then('valor_mensalidade')
    .otherwise(pl.col('valor_mensalidade') + 578).alias('mensalidade_titular')
]).select(pl.all().exclude('valor_mensalidade'))


In [204]:
final_optantes =  final_optantes.join(optantes_cru[['cpf_segurado', 'nome_segurado']], left_on='cpf_titular', right_on='cpf_segurado', how='inner')

In [199]:
dados_copart = dados_copart.select('cpf', 'nome_titular','emp_cod', 'matricula', 'cobranca_copart_titular', 'cobranca_copart_dependente')
dados_copart = dados_copart.with_columns([
    (pl.col('cobranca_copart_titular') + pl.col('cobranca_copart_dependente')).alias('total_cobranca'),
    pl.col('emp_cod').cast(pl.Int64)
]).filter(pl.col('total_cobranca') > 10).drop('total_cobranca')

In [213]:
excel_maida = (dados_copart
    .join(final_optantes, left_on='cpf', right_on='cpf_titular', how='full')
    .with_columns([
        pl.coalesce(pl.col('cpf'), pl.col('cpf_titular')).alias('cpf_titular'),
        pl.coalesce(pl.col('nome_titular'), pl.col('nome_segurado')).alias('nome'),
        pl.coalesce(pl.col('emp_cod'), pl.col('codigo_da_empresa')).alias('empresa'),
        pl.coalesce(pl.col('matricula'), pl.col('matricula_right')).alias('matricula'),
        pl.lit('Boleto').alias('forma_cobranca'),
        pl.col('mensalidade_titular').fill_null(0),
        pl.col('mensalidade_dependente').fill_null(0),
        pl.col('cobranca_copart_titular').fill_null(0),
        pl.col('cobranca_copart_dependente').fill_null(0),
        pl.col('situacao_funcional').fill_null('-')
    ])).select(['cpf_titular', 'nome', 'situacao_funcional', 'empresa', 'matricula', 
                'forma_cobranca', 'mensalidade_titular', 'mensalidade_dependente', 'cobranca_copart_titular', 'cobranca_copart_dependente'])

excel_maida = excel_maida.with_columns([
    (pl.col('mensalidade_titular') + pl.col('mensalidade_dependente') + pl.col('cobranca_copart_titular') + pl.col('cobranca_copart_dependente')).alias('Total_cobranca')
])


excel_maida.filter(pl.col('situacao_funcional') == 'Titular optante anterior')


cpf_titular,nome,situacao_funcional,empresa,matricula,forma_cobranca,mensalidade_titular,mensalidade_dependente,cobranca_copart_titular,cobranca_copart_dependente,Total_cobranca
str,str,str,i64,str,str,f64,i32,f64,f64,f64
"""002.246.941-97""","""GILNEY GUERRA DE MEDEIROS""","""Titular optante anterior""",552,"""01741950""","""Boleto""",578.0,0,399.56,0.0,977.56
"""201.827.751-00""","""MARIA IRIS MACENA MENEZES""","""Titular optante anterior""",802,"""69997225""","""Boleto""",578.0,250,199.57,0.43,1028.0
"""803.585.351-15""","""HILDEIJANETE RODRIGUES RIBEIRO…","""Titular optante anterior""",802,"""70214115""","""Boleto""",578.0,853,909.44,27.69,2368.13


In [217]:
excel_maida['cpf_titular'].n_unique()

5637